In [ ]:
# Step 0: 安裝基本套件
%pip install kagglehub
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install scikit-learn
%pip install wordcloud
%pip install transformers
%pip install datasets
%pip install torch
%pip install accelerate
%pip install hf_xet

In [ ]:
# Step 1: 下載到 dataset 到資料夾當中
import kagglehub
import os
import shutil

# kagglehub 下載到自己的快取
cached_path = kagglehub.dataset_download(
    "emineyetm/fake-news-detection-datasets"
)

# 複製到專案的 dataset 資料夾
target_path = os.path.normpath(os.path.join(os.getcwd(), "..", "dataset"))
os.makedirs(target_path, exist_ok=True)
shutil.copytree(cached_path, target_path, dirs_exist_ok=True)

print("Path to dataset files:", target_path)
print("Files:", os.listdir(target_path))

In [ ]:
# Step 2: 資料探索與清理
import os
import glob
import pandas as pd

# 找到 dataset 資料夾底下的 CSV 檔案 (支援子目錄)
data_root = os.path.normpath(os.path.join(os.getcwd(), '..', 'dataset'))
csv_paths = glob.glob(os.path.join(data_root, '**', '*.csv'), recursive=True)
print('Found CSV files under', data_root, ':')
for p in csv_paths:
    print('-', p)

# 讀取常見的 Fake / True 檔案，並標記 label (0: fake, 1: true)
dfs = []
for p in csv_paths:
    name = os.path.basename(p).lower()
    try:
        tmp = pd.read_csv(p)
    except Exception as e:
        print('Cannot read', p, '->', e)
        continue
    if 'fake' in name:
        tmp['label'] = 0
        dfs.append(tmp)
    elif 'true' in name:
        tmp['label'] = 1
        dfs.append(tmp)
# 若未自動辨識到 Fake/True，則取前兩個 CSV 檔並分別標記
if not dfs and len(csv_paths) >= 1:
    for i, p in enumerate(csv_paths[:2]):
        try:
            tmp = pd.read_csv(p)
        except Exception as e:
            print('Cannot read', p, '->', e)
            continue
        tmp['label'] = i
        dfs.append(tmp)

if not dfs:
    raise FileNotFoundError(f'No readable CSV files found under {data_root}')

# 合併資料並顯示基本資訊
df = pd.concat(dfs, ignore_index=True, sort=False)
print('Combined dataframe shape:', df.shape)
print('Columns:', df.columns.tolist())

# 偵測文字欄位 (優先順序)
candidate_text_cols = ['text', 'content', 'article', 'statement', 'title', 'headline']
text_col = None
for c in candidate_text_cols:
    if c in df.columns:
        text_col = c
        break
if text_col is None:
    # 若沒有常見名稱，嘗試找出 dtype 為 object 的欄位
    for c in df.columns:
        if df[c].dtype == 'object':
            text_col = c
            break

if text_col is None:
    raise ValueError('Cannot find a text-like column to use for preprocessing. Columns: ' + str(df.columns.tolist()))

print('Using text column:', text_col)

# 檢查缺失值與重複值
missing_count = df[text_col].isna().sum()
dup_count = df.duplicated(subset=[text_col]).sum()
print(f'Missing in {text_col}:', missing_count)
print(f'Duplicated {text_col}:', dup_count)

# 將欄位轉為字串並去除前後空白
df[text_col] = df[text_col].astype(str).str.strip()

# 移除空字串或 NaN 的列
before = len(df)
df = df[df[text_col].astype(bool)]  # remove empty strings and NaN-as-str('nan')
after_drop_empty = len(df)
print('Dropped empty/missing rows:', before - after_drop_empty)

# 移除基於文字內容的重複列 (保留第一筆)
before_dup = len(df)
df = df.drop_duplicates(subset=[text_col])
after_dup = len(df)
print('Dropped duplicate rows:', before_dup - after_dup)

# 重設索引並保留必要欄位 (至少 text_col 與 label)
df = df.reset_index(drop=True)
cols_to_keep = [c for c in [text_col, 'label'] if c in df.columns]
clean_df = df[cols_to_keep].copy()

# 儲存清理後的結果
out_path = os.path.join(data_root, 'cleaned_news.csv')
clean_df.to_csv(out_path, index=False)
print('Saved cleaned data to', out_path)

# 顯示前幾筆與類別分布
display(clean_df.head())
if 'label' in clean_df.columns:
    print('Label counts:')
    print(clean_df['label'].value_counts())

In [ ]:
# Step 3: 文本量化分析（加強版）
# 原有：word_count, sentence_count, avg_word_len, TTR
# 新增：情感分析 (VADER)、可讀性 (Flesch-Kincaid)、標點密度、大寫比例

import os
import re
import subprocess
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

for pkg in ['textstat', 'vaderSentiment']:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

import textstat
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sns.set(style='whitegrid')
data_root = os.path.normpath(os.path.join(os.getcwd(), '..', 'dataset'))
plots_dir = os.path.join(data_root, 'plots')
os.makedirs(plots_dir, exist_ok=True)

df = pd.read_csv(os.path.join(data_root, 'cleaned_news.csv'))
df['label_name'] = df['label'].map({0: 'Fake', 1: 'Real'})

# ── 原有特徵 ──────────────────────────────────────────────────────────────────
df['char_count']     = df['text'].str.len()
df['word_count']     = df['text'].str.split().str.len()
df['sentence_count'] = df['text'].apply(lambda t: max(1, len(re.split(r'[.!?]+', t))))
df['avg_word_len']   = df['text'].apply(
    lambda t: np.mean([len(w) for w in t.split()]) if t.split() else 0
)
df['ttr'] = df['text'].apply(
    lambda t: len(set(t.lower().split())) / max(1, len(t.split()))
)

# ── 新增：標點密度（每個詞平均出現幾個驚嘆/問號）────────────────────────────────
df['exclaim_density']  = df['text'].apply(lambda t: t.count('!') / max(1, len(t.split())))
df['question_density'] = df['text'].apply(lambda t: t.count('?') / max(1, len(t.split())))

# ── 新增：大寫比例（全大寫且長度 > 1 的單字佔比）────────────────────────────────
def caps_ratio(text):
    words = text.split()
    if not words:
        return 0.0
    return sum(1 for w in words if w.isupper() and len(w) > 1) / len(words)

df['caps_ratio'] = df['text'].apply(caps_ratio)

# ── 新增：Flesch-Kincaid 可讀性分數 ─────────────────────────────────────────
# 分數越高越易讀；假新聞傾向用語通俗或刻意簡化，預期分數偏高
df['flesch_score'] = df['text'].apply(lambda t: textstat.flesch_reading_ease(t[:5000]))

# ── 新增：VADER 情感分析 ──────────────────────────────────────────────────────
# compound: -1（極負面）~ +1（極正面）；假新聞情緒通常更極端（絕對值較大）
vader = SentimentIntensityAnalyzer()

def get_sentiment(text):
    s = vader.polarity_scores(text[:3000])
    return s['compound'], s['pos'], s['neg']

print('Running VADER sentiment analysis (may take a moment)...')
sent_vals        = df['text'].apply(get_sentiment)
df['vader_compound'] = sent_vals.apply(lambda x: x[0])
df['vader_pos']      = sent_vals.apply(lambda x: x[1])
df['vader_neg']      = sent_vals.apply(lambda x: x[2])
df['vader_abs']      = df['vader_compound'].abs()

# ── 統計摘要 ──────────────────────────────────────────────────────────────────
summary_cols = ['word_count', 'sentence_count', 'avg_word_len', 'ttr',
                'exclaim_density', 'caps_ratio', 'flesch_score', 'vader_compound', 'vader_abs']
print(df.groupby('label_name')[summary_cols].describe().round(3))

palette = {'Fake': '#e05c5c', 'Real': '#4c9be8'}

# ── 視覺化 1：原有特徵分布 ────────────────────────────────────────────────────
orig_features = {
    'word_count'    : 'Word Count',
    'sentence_count': 'Sentence Count',
    'avg_word_len'  : 'Avg Word Length',
    'ttr'           : 'Type-Token Ratio',
}
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (col, title) in zip(axes.flatten(), orig_features.items()):
    for label, grp in df.groupby('label_name'):
        ax.hist(grp[col].clip(upper=grp[col].quantile(0.99)),
                bins=60, alpha=0.55, label=label, color=palette[label], density=True)
    ax.set_title(f'Distribution of {title}')
    ax.set_xlabel(title); ax.set_ylabel('Density'); ax.legend()
plt.suptitle('Text Quantitative Analysis — Fake vs. Real News', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'step3_text_distribution.png'), bbox_inches='tight')
plt.show()

# ── 視覺化 2：新增特徵分布 ────────────────────────────────────────────────────
new_features = {
    'exclaim_density': 'Exclamation Density',
    'caps_ratio'     : 'ALL-CAPS Word Ratio',
    'flesch_score'   : 'Flesch Reading Ease',
    'vader_compound' : 'VADER Sentiment (Compound)',
}
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (col, title) in zip(axes.flatten(), new_features.items()):
    for label, grp in df.groupby('label_name'):
        vals = grp[col].clip(lower=grp[col].quantile(0.01), upper=grp[col].quantile(0.99))
        ax.hist(vals, bins=60, alpha=0.55, label=label, color=palette[label], density=True)
    ax.set_title(f'Distribution of {title}')
    ax.set_xlabel(title); ax.set_ylabel('Density'); ax.legend()
plt.suptitle('Enhanced Text Features — Fake vs. Real News', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'step3_enhanced_features.png'), bbox_inches='tight')
plt.show()
print('Saved ->', os.path.join(plots_dir, 'step3_enhanced_features.png'))

# ── 視覺化 3：六項特徵 Boxplot 比較 ──────────────────────────────────────────
box_features = ['word_count', 'ttr', 'exclaim_density', 'caps_ratio', 'flesch_score', 'vader_abs']
box_titles   = ['Word Count', 'TTR', 'Exclaim Density', 'CAPS Ratio', 'Flesch Score', 'Sentiment |Compound|']
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, col, title in zip(axes.flatten(), box_features, box_titles):
    sns.boxplot(data=df, x='label_name', y=col, palette=palette, ax=ax,
                order=['Fake', 'Real'], showfliers=False)
    ax.set_title(title); ax.set_xlabel('')
plt.suptitle('Feature Comparison — Boxplot (no outliers)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'step3_boxplot_all.png'), bbox_inches='tight')
plt.show()
print('Saved ->', os.path.join(plots_dir, 'step3_boxplot_all.png'))

# ── 視覺化 4：標籤分布 ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
counts = df['label_name'].value_counts()
ax.bar(counts.index, counts.values, color=[palette[l] for l in counts.index], edgecolor='white')
for i, v in enumerate(counts.values):
    ax.text(i, v + 100, f'{v:,}', ha='center', fontsize=11)
ax.set_title('Class Distribution'); ax.set_ylabel('Count'); ax.set_xlabel('Label')
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'step3_label_distribution.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Step 4: 關鍵詞與主題建模（加強版）
# 1. TF-IDF 高頻詞分析（Unigram Top-30 + Bigram Top-30）
# 2. WordCloud 視覺化
# 3. LDA 主題模型（各 5 個主題）
# 4. 命名實體頻率分析（NER）— Person / Organization / Location 在假/真新聞的出現差異

import os
import sys
import subprocess
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

for pkg, import_name in [('wordcloud', 'wordcloud'), ('pyldavis', 'pyLDAvis')]:
    try:
        __import__(import_name)
    except ImportError:
        print(f'{pkg} not found, installing...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

# spaCy for NER
SPACY_AVAILABLE = False
try:
    import spacy
    nlp = spacy.load('en_core_web_sm')
    SPACY_AVAILABLE = True
    print('spaCy en_core_web_sm loaded.')
except Exception:
    try:
        print('Installing spaCy and en_core_web_sm model...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'spacy'])
        subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])
        import spacy
        nlp = spacy.load('en_core_web_sm')
        SPACY_AVAILABLE = True
        print('spaCy ready.')
    except Exception as e:
        print(f'spaCy unavailable, NER section will be skipped: {e}')

from wordcloud import WordCloud
import pyLDAvis
import pyLDAvis.lda_model

sns.set(style='whitegrid')
data_root = os.path.normpath(os.path.join(os.getcwd(), '..', 'dataset'))
plots_dir = os.path.join(data_root, 'plots')
os.makedirs(plots_dir, exist_ok=True)

df = pd.read_csv(os.path.join(data_root, 'cleaned_news.csv'))
fake_texts = df[df['label'] == 0]['text'].astype(str).tolist()
real_texts = df[df['label'] == 1]['text'].astype(str).tolist()

palette = {'Fake': '#e05c5c', 'Real': '#4c9be8'}

# ─────────────────────────────────────────────────────────────────────────────
# 1. TF-IDF — Unigram + Bigram Top-30
# ─────────────────────────────────────────────────────────────────────────────
def top_tfidf_words(texts, n=30, ngram_range=(1, 1)):
    vec = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=ngram_range)
    X = vec.fit_transform(texts)
    scores = np.asarray(X.mean(axis=0)).flatten()
    words  = np.array(vec.get_feature_names_out())
    idx    = np.argsort(scores)[::-1][:n]
    return pd.DataFrame({'word': words[idx], 'score': scores[idx]})

top_fake_uni = top_tfidf_words(fake_texts, ngram_range=(1, 1))
top_real_uni = top_tfidf_words(real_texts, ngram_range=(1, 1))
top_fake_bi  = top_tfidf_words(fake_texts, ngram_range=(2, 2))
top_real_bi  = top_tfidf_words(real_texts, ngram_range=(2, 2))

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, df_top, title, color in zip(
    axes, [top_fake_uni, top_real_uni],
    ['Top-30 Unigram — Fake News', 'Top-30 Unigram — Real News'],
    ['#e05c5c', '#4c9be8']
):
    sns.barplot(data=df_top, y='word', x='score', ax=ax, color=color)
    ax.set_title(title, fontsize=12); ax.set_xlabel('Mean TF-IDF Score'); ax.set_ylabel('')
plt.suptitle('TF-IDF Unigram Keywords', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'step4_tfidf_unigram.png'), bbox_inches='tight')
plt.show()

# Bigram 圖：比 unigram 更能揭示假新聞慣用詞組（如 "deep state"、"fake news"）
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, df_top, title, color in zip(
    axes, [top_fake_bi, top_real_bi],
    ['Top-30 Bigram — Fake News', 'Top-30 Bigram — Real News'],
    ['#e05c5c', '#4c9be8']
):
    sns.barplot(data=df_top, y='word', x='score', ax=ax, color=color)
    ax.set_title(title, fontsize=12); ax.set_xlabel('Mean TF-IDF Score'); ax.set_ylabel('')
plt.suptitle('TF-IDF Bigram Keywords (Phrases)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'step4_tfidf_bigram.png'), bbox_inches='tight')
plt.show()
print('Saved TF-IDF unigram & bigram plots')

# ─────────────────────────────────────────────────────────────────────────────
# 2. WordCloud
# ─────────────────────────────────────────────────────────────────────────────
def make_wordcloud(texts, title, colormap, save_path):
    combined = ' '.join(texts)
    wc = WordCloud(width=900, height=500, background_color='white',
                   colormap=colormap, max_words=200, collocations=False).generate(combined)
    plt.figure(figsize=(12, 6))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off'); plt.title(title, fontsize=14)
    plt.tight_layout(); plt.savefig(save_path, bbox_inches='tight'); plt.show()
    print('Saved ->', save_path)

make_wordcloud(fake_texts, 'WordCloud — Fake News', 'Reds',
               os.path.join(plots_dir, 'step4_wordcloud_fake.png'))
make_wordcloud(real_texts, 'WordCloud — Real News', 'Blues',
               os.path.join(plots_dir, 'step4_wordcloud_real.png'))

# ─────────────────────────────────────────────────────────────────────────────
# 3. LDA 主題模型
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.decomposition import LatentDirichletAllocation

N_TOPICS, N_TOP = 5, 10

def run_lda(texts, n_topics=N_TOPICS, max_features=3000, sample_n=5000):
    if len(texts) > sample_n:
        rng = np.random.default_rng(42)
        texts = [texts[i] for i in rng.choice(len(texts), sample_n, replace=False)]
    cv  = CountVectorizer(max_features=max_features, stop_words='english', min_df=3)
    dtm = cv.fit_transform(texts)
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=42,
                                    max_iter=15, learning_method='online')
    lda.fit(dtm)
    vocab  = np.array(cv.get_feature_names_out())
    topics = [{'topic': i + 1, 'words': ', '.join(vocab[np.argsort(comp)[::-1][:N_TOP]])}
              for i, comp in enumerate(lda.components_)]
    return lda, cv, pd.DataFrame(topics)

print('\n=== LDA on Fake News ===')
lda_fake, cv_fake, topics_fake = run_lda(fake_texts)
display(topics_fake)

print('\n=== LDA on Real News ===')
lda_real, cv_real, topics_real = run_lda(real_texts)
display(topics_real)

topics_fake['label'] = 'Fake'
topics_real['label'] = 'Real'
pd.concat([topics_fake, topics_real], ignore_index=True).to_csv(
    os.path.join(data_root, 'lda_topics.csv'), index=False)

def plot_topic_heatmap(lda, cv, title, save_path, n_words=10):
    vocab   = np.array(cv.get_feature_names_out())
    top_idx = sorted(set(
        idx for comp in lda.components_
        for idx in np.argsort(comp)[::-1][:n_words].tolist()
    ))
    sub = lda.components_[:, top_idx]
    sub = sub / sub.sum(axis=1, keepdims=True)
    df_heat = pd.DataFrame(sub, columns=vocab[top_idx],
                           index=[f'Topic {i+1}' for i in range(lda.n_components)])
    plt.figure(figsize=(max(10, n_words * 0.9), 4))
    sns.heatmap(df_heat, cmap='YlOrRd', linewidths=0.3)
    plt.title(title, fontsize=12); plt.ylabel('Topic')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout(); plt.savefig(save_path, bbox_inches='tight'); plt.show()
    print('Saved ->', save_path)

plot_topic_heatmap(lda_fake, cv_fake, 'LDA Topic-Word Heatmap — Fake News',
                   os.path.join(plots_dir, 'step4_lda_heatmap_fake.png'))
plot_topic_heatmap(lda_real, cv_real, 'LDA Topic-Word Heatmap — Real News',
                   os.path.join(plots_dir, 'step4_lda_heatmap_real.png'))

def save_ldavis(lda, cv, texts, save_path, sample_n=5000):
    if len(texts) > sample_n:
        rng = np.random.default_rng(42)
        texts = [texts[i] for i in rng.choice(len(texts), sample_n, replace=False)]
    dtm = cv.transform(texts)
    try:
        vis = pyLDAvis.lda_model.prepare(lda, dtm, cv, mds='mmds')
        pyLDAvis.save_html(vis, save_path)
        print('Saved interactive LDA ->', save_path)
    except Exception as e:
        print('pyLDAvis failed (non-critical):', e)

save_ldavis(lda_fake, cv_fake, fake_texts, os.path.join(data_root, 'ldavis_fake.html'))
save_ldavis(lda_real, cv_real, real_texts, os.path.join(data_root, 'ldavis_real.html'))

# ─────────────────────────────────────────────────────────────────────────────
# 4. 命名實體頻率分析（NER）
# 比較哪些 人名 / 組織 / 地名 在假/真新聞中最常被提及
# ─────────────────────────────────────────────────────────────────────────────
if SPACY_AVAILABLE:
    def extract_entities(texts, sample_n=3000, entity_types=('PERSON', 'ORG', 'GPE')):
        if len(texts) > sample_n:
            rng = np.random.default_rng(42)
            texts = [texts[i] for i in rng.choice(len(texts), sample_n, replace=False)]
        counter = {t: Counter() for t in entity_types}
        for text in texts:
            doc = nlp(text[:1000], disable=['parser', 'tagger', 'lemmatizer'])
            for ent in doc.ents:
                if ent.label_ in counter:
                    counter[ent.label_][ent.text.strip()] += 1
        return counter

    print('\nExtracting named entities (may take a minute)...')
    ents_fake = extract_entities(fake_texts)
    ents_real = extract_entities(real_texts)

    entity_labels = {'PERSON': 'Person', 'ORG': 'Organization', 'GPE': 'Location'}
    fig, axes = plt.subplots(len(entity_labels), 2, figsize=(14, 4 * len(entity_labels)))

    for row, (etype, ename) in enumerate(entity_labels.items()):
        for col, (ents, label, color) in enumerate([
            (ents_fake, 'Fake', '#e05c5c'),
            (ents_real, 'Real', '#4c9be8')
        ]):
            top = pd.DataFrame(ents[etype].most_common(15), columns=['entity', 'count'])
            ax = axes[row][col]
            sns.barplot(data=top, y='entity', x='count', ax=ax, color=color)
            ax.set_title(f'Top {ename}s — {label} News', fontsize=11)
            ax.set_xlabel('Mention Count'); ax.set_ylabel('')

    plt.suptitle('Named Entity Frequency: Fake vs. Real News', fontsize=14)
    plt.tight_layout()
    ner_path = os.path.join(plots_dir, 'step4_ner_frequency.png')
    plt.savefig(ner_path, bbox_inches='tight'); plt.show()
    print('Saved NER frequency chart ->', ner_path)

    ner_rows = [
        {'label': label, 'entity_type': etype, 'entity': ent, 'count': cnt}
        for label, ents_dict in [('Fake', ents_fake), ('Real', ents_real)]
        for etype in entity_labels
        for ent, cnt in ents_dict[etype].most_common(30)
    ]
    pd.DataFrame(ner_rows).to_csv(os.path.join(data_root, 'ner_frequency.csv'), index=False)
    print('Saved NER CSV ->', os.path.join(data_root, 'ner_frequency.csv'))
else:
    print('NER section skipped (spaCy not available).')

In [ ]:
# Step 5: 訓練 BERT 模型與傳統模型比較
import os
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Hugging Face imports
from datasets import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments

# 偵測 CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

data_root = os.path.normpath(os.path.join(os.getcwd(), '..', 'dataset'))
df = pd.read_csv(os.path.join(data_root, 'cleaned_news.csv'))
if 'label' not in df.columns:
    raise ValueError('cleaned_news.csv 必須包含 label 欄位')

# 取樣以加速示範（如資料量大，可調整 sample_n 或設為 None 表示使用全部）
sample_n = 8000  # set to None to use all rows
if sample_n is not None and len(df) > sample_n:
    df = df.sample(n=sample_n, random_state=42)

texts = df['text'].astype(str).tolist()
labels = df['label'].astype(int).tolist()

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, stratify=labels, random_state=42)

# Baseline: TF-IDF + RandomForest
tfidf = TfidfVectorizer(max_features=20000, stop_words='english')
Xtr_tfidf = tfidf.fit_transform(X_train)
Xte_tfidf = tfidf.transform(X_test)
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(Xtr_tfidf, y_train)
y_pred_rf = rf.predict(Xte_tfidf)
rf_acc = accuracy_score(y_test, y_pred_rf)
rf_prec = precision_score(y_test, y_pred_rf, average='binary')
rf_rec = recall_score(y_test, y_pred_rf, average='binary')
rf_f1 = f1_score(y_test, y_pred_rf, average='binary')
print('RandomForest Results:')
print(f'  Accuracy: {rf_acc:.4f}, Precision: {rf_prec:.4f}, Recall: {rf_rec:.4f}, F1: {rf_f1:.4f}')
print(classification_report(y_test, y_pred_rf))

# BERT (DistilBERT) fine-tuning -- 使用 CUDA 加速
model_name = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)

train_df = pd.DataFrame({'text': X_train, 'label': y_train})
test_df = pd.DataFrame({'text': X_test, 'label': y_test})
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)
train_ds = train_ds.map(tokenize, batched=True, remove_columns=['text'])
test_ds = test_ds.map(tokenize, batched=True, remove_columns=['text'])
train_ds = train_ds.rename_column('label', 'labels')
test_ds = test_ds.rename_column('label', 'labels')
train_ds.set_format('torch')
test_ds.set_format('torch')

model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)
model = model.to(device)

use_cuda = torch.cuda.is_available()
training_args = TrainingArguments(
    output_dir=os.path.join(data_root, 'bert_out'),
    per_device_train_batch_size=32 if use_cuda else 16,   # GPU 可用較大 batch
    per_device_eval_batch_size=64 if use_cuda else 32,
    num_train_epochs=2,
    logging_steps=50,
    fp16=use_cuda,            # 半精度浮點數加速（僅 CUDA）
    dataloader_pin_memory=use_cuda,
    use_cpu=not use_cuda,
)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, average='binary'),
        'recall': recall_score(labels, preds, average='binary'),
        'f1': f1_score(labels, preds, average='binary')
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)
print(f'Starting BERT training on {device} (this may take several minutes)')
trainer.train()
res = trainer.evaluate()
print('BERT evaluation:', res)

# 儲存比較結果
results = {
    'model': ['RandomForest', 'DistilBERT'],
    'accuracy': [rf_acc, res.get('eval_accuracy')],
    'precision': [rf_prec, res.get('eval_precision')],
    'recall': [rf_rec, res.get('eval_recall')],
    'f1': [rf_f1, res.get('eval_f1')]
}
pd.DataFrame(results).to_csv(os.path.join(data_root, 'model_comparison_results.csv'), index=False)
print('Saved model comparison to', os.path.join(data_root, 'model_comparison_results.csv'))

In [ ]:
# Step 6: 完整的測試報告
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

sns.set(style='whitegrid')
data_root = os.path.normpath(os.path.join(os.getcwd(), '..', 'dataset'))
plots_dir = os.path.join(data_root, 'plots')
os.makedirs(plots_dir, exist_ok=True)

# Load cleaned data and prepare test set (match Step5 sampling)
df = pd.read_csv(os.path.join(data_root, 'cleaned_news.csv'))
sample_n = 8000
if sample_n is not None and len(df) > sample_n:
    df = df.sample(n=sample_n, random_state=42)
texts = df['text'].astype(str).tolist()
labels = df['label'].astype(int).tolist()
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, stratify=labels, random_state=42)

# Ensure TF-IDF + RF available (reuse if in kernel, else train a quick baseline)
try:
    tfidf
    rf
    Xte_tfidf
    print('Using TF-IDF + RF from kernel.')
except NameError:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.ensemble import RandomForestClassifier
    print('Training TF-IDF + RandomForest for reporting...')
    tfidf = TfidfVectorizer(max_features=20000, stop_words='english')
    Xtr_tfidf = tfidf.fit_transform(X_train)
    Xte_tfidf = tfidf.transform(X_test)
    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(Xtr_tfidf, y_train)

# Evaluate RandomForest
y_pred_rf = rf.predict(Xte_tfidf)
y_prob_rf = rf.predict_proba(Xte_tfidf) if hasattr(rf, 'predict_proba') else None
rf_acc = accuracy_score(y_test, y_pred_rf)
rf_prec = precision_score(y_test, y_pred_rf, average='binary')
rf_rec = recall_score(y_test, y_pred_rf, average='binary')
rf_f1 = f1_score(y_test, y_pred_rf, average='binary')
print('RandomForest — Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1: {:.4f}'.format(rf_acc, rf_prec, rf_rec, rf_f1))
print(classification_report(y_test, y_pred_rf))

# Confusion matrix (RF)
cm_rf = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(5,4))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', xticklabels=['fake','real'], yticklabels=['fake','real'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — RandomForest')
rf_cm_path = os.path.join(plots_dir, f'rf_confusion_matrix_{int(time.time())}.png')
plt.tight_layout()
plt.savefig(rf_cm_path, bbox_inches='tight')
plt.show()
print('Saved RF confusion matrix to', rf_cm_path)

# Try to evaluate BERT (if available as pipeline or trainer)
bert_metrics = None
try:
    if 'bert_clf' in globals():
        bert_preds = []
        batch = 32
        for i in range(0, len(X_test), batch):
            batch_texts = X_test[i:i+batch]
            res = bert_clf(batch_texts, truncation=True, max_length=256)
            for r in res:
                lab = r.get('label')
                if isinstance(lab, str) and lab.startswith('LABEL_'):
                    bert_preds.append(int(lab.split('_')[-1]))
                else:
                    try:
                        bert_preds.append(int(lab))
                    except Exception:
                        bert_preds.append(1 if r.get('score',0) > 0.5 else 0)
        if bert_preds:
            bert_acc = accuracy_score(y_test, bert_preds)
            bert_prec = precision_score(y_test, bert_preds, average='binary')
            bert_rec = recall_score(y_test, bert_preds, average='binary')
            bert_f1 = f1_score(y_test, bert_preds, average='binary')
            print('BERT — Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1: {:.4f}'.format(bert_acc, bert_prec, bert_rec, bert_f1))
            print(classification_report(y_test, bert_preds))
            cm_bert = confusion_matrix(y_test, bert_preds)
            plt.figure(figsize=(5,4))
            sns.heatmap(cm_bert, annot=True, fmt='d', cmap='Greens', xticklabels=['fake','real'], yticklabels=['fake','real'])
            plt.xlabel('Predicted')
            plt.ylabel('Actual')
            plt.title('Confusion Matrix — BERT')
            bert_cm_path = os.path.join(plots_dir, f'bert_confusion_matrix_{int(time.time())}.png')
            plt.tight_layout()
            plt.savefig(bert_cm_path, bbox_inches='tight')
            plt.show()
            print('Saved BERT confusion matrix to', bert_cm_path)
            bert_metrics = {'model':'BERT', 'accuracy':bert_acc, 'precision':bert_prec, 'recall':bert_rec, 'f1':bert_f1}
except Exception as e:
    print('BERT evaluation skipped or failed:', e)

# Save metrics to CSV
metrics = [{'model':'RandomForest', 'accuracy':rf_acc, 'precision':rf_prec, 'recall':rf_rec, 'f1':rf_f1}]
if bert_metrics is not None:
    metrics.append(bert_metrics)
out_csv = os.path.join(data_root, 'test_report_metrics.csv')
pd.DataFrame(metrics).to_csv(out_csv, index=False)
print('Saved test metrics to', out_csv)

# Also save confusion matrices as separate images (paths already printed)


In [ ]:
# Step 7: 模型可解釋性（LIME） - 顯示模型在判斷「假」時看重的關鍵字
# 這個 cell 會：
# - 使用 LIME 的 LimeTextExplainer 對已訓練的 RandomForest (TF-IDF 輸入) 做局部解釋
# - 對幾個被模型預測為 "fake" 的樣本產生 word-level 解釋
# - 繪製並儲存每個樣本的條形圖，以及彙整的 top 關鍵字圖表

import os
import sys
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier

# 嘗試 import lime，若無則安裝
try:
    from lime.lime_text import LimeTextExplainer
except Exception:
    print('lime not found, installing...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lime'])
    from lime.lime_text import LimeTextExplainer

sns.set(style='whitegrid')

# 載入資料
data_root = os.path.normpath(os.path.join(os.getcwd(), '..', 'dataset'))
df = pd.read_csv(os.path.join(data_root, 'cleaned_news.csv'))

# 使用與 Step5 / Step6 相同的 sample 與 split 設定
sample_n = 8000
if sample_n is not None and len(df) > sample_n:
    df = df.sample(n=sample_n, random_state=42)
texts = df['text'].astype(str).tolist()
labels = df['label'].astype(int).tolist()

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, stratify=labels, random_state=42)

# 若 workspace 中已有訓練好的 tfidf 與 rf（從 Step6 執行過來），則使用；否則快速訓練一個 baseline
need_train = False
try:
    tfidf
    rf
except NameError:
    need_train = True

if need_train:
    print('Training TF-IDF + RandomForest baseline for explanations...')
    tfidf = TfidfVectorizer(max_features=20000, stop_words='english')
    Xtr_tfidf = tfidf.fit_transform(X_train)
    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(Xtr_tfidf, y_train)
else:
    # 若已存在，確保 Xte_tfidf 可用
    try:
        Xte_tfidf
    except NameError:
        Xtr_tfidf = tfidf.transform(X_train)
        Xte_tfidf = tfidf.transform(X_test)

# 定義 predict_proba callable，接受 raw texts 並回傳 [P(fake), P(real)]
def predict_proba(texts_list):
    Xv = tfidf.transform(texts_list)
    probs = rf.predict_proba(Xv)
    # 確保 columns 依照 class 0 (fake), 1 (real)
    classes = list(rf.classes_)
    if classes == [0, 1]:
        return probs
    else:
        order = [classes.index(0), classes.index(1)]
        return probs[:, order]

# 建立 LIME explainer
explainer = LimeTextExplainer(class_names=['fake', 'real'])

# 找到被模型預測為 fake 的範例（最多 5 個）
y_pred_rf = rf.predict(Xte_tfidf)
idx_fake = [i for i, p in enumerate(y_pred_rf) if p == 0]
if not idx_fake:
    # 若沒有直接被預測為 fake，選擇對 fake 機率最高的幾筆
    probs_all = predict_proba(X_test)
    fake_probs = probs_all[:, 0]
    idx_fake = list(np.argsort(fake_probs)[::-1][:5])
else:
    idx_fake = idx_fake[:5]

# 確保 plots 資料夾存在
os.makedirs(os.path.join(data_root, 'plots'), exist_ok=True)

explanations = []
for j, test_idx in enumerate(idx_fake):
    sample_text = X_test[test_idx]
    print(f'Explaining example index {test_idx} (predicted={y_pred_rf[test_idx]})')
    try:
        exp = explainer.explain_instance(sample_text, predict_proba, num_features=15, top_labels=1)
        # 取得對 class 'fake' (label 0) 的解釋
        feat_list = exp.as_list(label=0)
    except Exception as e:
        print('LIME failed for this instance:', e)
        feat_list = []

    df_exp = pd.DataFrame(feat_list, columns=['feature', 'weight'])
    df_exp['example_idx'] = test_idx
    df_exp['text_sample'] = sample_text
    explanations.append(df_exp)

    # 畫圖並儲存
    if not df_exp.empty:
        plt.figure(figsize=(6, max(3, 0.35 * len(df_exp))))
        sns.barplot(x='weight', y='feature', data=df_exp, palette='vlag')
        plt.title(f'LIME features for example idx {test_idx} (pred=fake)')
        plt.xlabel('Weight (positive -> evidence for this class)')
        plt.tight_layout()
        figfile = os.path.join(data_root, 'plots', f'lime_explain_example_{j}_idx_{test_idx}.png')
        plt.savefig(figfile, bbox_inches='tight')
        plt.show()
        print('Saved explanation figure to', figfile)

# 合併並儲存所有解釋結果
if explanations:
    all_exp = pd.concat(explanations, ignore_index=True)
    all_exp.to_csv(os.path.join(data_root, 'lime_explanations_examples.csv'), index=False)
    print('Saved LIME explanations CSV to', os.path.join(data_root, 'lime_explanations_examples.csv'))

    # 彙整 top 關鍵字（以平均絕對權重排序）
    agg = all_exp.groupby('feature')['weight'].apply(lambda x: np.mean(np.abs(x))).sort_values(ascending=False).head(30)
    plt.figure(figsize=(6, max(4, 0.25 * len(agg))))
    sns.barplot(x=agg.values, y=agg.index, palette='magma')
    plt.title('Top contributing words (avg |weight|) for examples predicted as fake')
    plt.xlabel('Avg |weight|')
    plt.tight_layout()
    agg_fig = os.path.join(data_root, 'plots', 'lime_top_agg_fake.png')
    plt.savefig(agg_fig, bbox_inches='tight')
    plt.show()
    print('Saved aggregated LIME plot to', agg_fig)
else:
    print('No explanations generated (no examples or LIME failed).')


In [ ]:
# Step 8: 新聞測試 - 對新的新聞文本進行預測並解釋
# 使用說明：將 `news_text` 變數設為你想測試的新聞文字，執行此 cell。
# 會輸出 RandomForest (TF-IDF) 的預測與機率，以及（若有）BERT 的預測；並用 LIME 給出解釋。

import os
import sys
import time
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

sns.set(style='whitegrid')

data_root = os.path.normpath(os.path.join(os.getcwd(), '..', 'dataset'))
plots_dir = os.path.join(data_root, 'plots')
os.makedirs(plots_dir, exist_ok=True)

# 確保 LIME 可用
try:
    from lime.lime_text import LimeTextExplainer
except Exception:
    print('Installing lime...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lime'])
    from lime.lime_text import LimeTextExplainer

# 嘗試載入已訓練的 TF-IDF + RF（若在記憶體中），否則重新訓練一個小的 baseline
try:
    tfidf
    rf
    Xte_tfidf
    X_test
    print('Found existing TF-IDF + RF in kernel.')
except NameError:
    print('No TF-IDF/RF in kernel. Training a quick baseline using cleaned_news.csv...')
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    df = pd.read_csv(os.path.join(data_root, 'cleaned_news.csv'))
    sample_n = 8000
    if sample_n is not None and len(df) > sample_n:
        df = df.sample(n=sample_n, random_state=42)
    texts = df['text'].astype(str).tolist()
    labels = df['label'].astype(int).tolist()
    X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, stratify=labels, random_state=42)
    tfidf = TfidfVectorizer(max_features=20000, stop_words='english')
    Xtr_tfidf = tfidf.fit_transform(X_train)
    Xte_tfidf = tfidf.transform(X_test)
    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(Xtr_tfidf, y_train)

# 載入（若可用）BERT pipeline — 改為搜尋子資料夾中含有效 config.json 的 checkpoint
bert_root = os.path.join(data_root, 'bert_out')

def find_bert_dir(root):
    if not os.path.isdir(root):
        return None
    cfg = os.path.join(root, 'config.json')
    if os.path.exists(cfg):
        try:
            j = json.load(open(cfg))
            if 'model_type' in j:
                return root
        except Exception:
            pass
    for sub in sorted(os.listdir(root), reverse=True):
        p = os.path.join(root, sub)
        if os.path.isdir(p):
            cfg = os.path.join(p, 'config.json')
            if os.path.exists(cfg):
                try:
                    j = json.load(open(cfg))
                    if 'model_type' in j:
                        return p
                except Exception:
                    continue
    return None

bert_dir = find_bert_dir(bert_root)
bert_available = bool(bert_dir)
if bert_available:
    try:
        from transformers import pipeline
        device = 0 if __import__('torch').cuda.is_available() else -1
        bert_clf = pipeline('text-classification', model=bert_dir, device=device)
        print('Loaded BERT pipeline from', bert_dir)
    except Exception as e:
        print('Failed to load BERT pipeline:', e)
        bert_available = False
else:
    print('No valid BERT model found under', bert_root)

# predict_and_explain function
from sklearn.metrics import accuracy_score

def predict_and_explain(text, explain=True, top_k_features=15, save=True):
    """返回 dict：包含 rf_pred, rf_prob_fake, bert_pred (若可用), bert_score (若可用), explanation_path (若 explain)
    """
    out = {'text': text}
    # RF prediction
    Xv = tfidf.transform([text])
    probs = rf.predict_proba(Xv)
    # Classes order
    classes = list(rf.classes_)
    if classes == [0, 1]:
        prob_fake = float(probs[0,0])
        prob_real = float(probs[0,1])
    else:
        order = [classes.index(0), classes.index(1)]
        prob_fake = float(probs[0, order[0]])
        prob_real = float(probs[0, order[1]])
    rf_pred = int(rf.predict(Xv)[0])
    out.update({'rf_pred': int(rf_pred), 'rf_prob_fake': prob_fake, 'rf_prob_real': prob_real})
    print(f'RF prediction: {rf_pred} (0=fake,1=real), P(fake)={prob_fake:.4f}, P(real)={prob_real:.4f}')

    # BERT prediction (if available)
    if bert_available:
        try:
            res = bert_clf(text, truncation=True, max_length=256)
            lab = res[0]['label']
            score = float(res[0].get('score', 0.0))
            if lab.startswith('LABEL_'):
                bert_pred = int(lab.split('_')[-1])
            else:
                try:
                    bert_pred = int(lab)
                except Exception:
                    bert_pred = 1 if score > 0.5 else 0
            out.update({'bert_pred': bert_pred, 'bert_score': score})
            print(f'BERT prediction: {bert_pred}, score={score:.4f}')
        except Exception as e:
            print('BERT prediction failed:', e)
    else:
        out.update({'bert_pred': None, 'bert_score': None})

    explanation_path = None
    if explain:
        explainer = LimeTextExplainer(class_names=['fake', 'real'])
        try:
            exp = explainer.explain_instance(text, lambda x: rf.predict_proba(tfidf.transform(x)), num_features=top_k_features, top_labels=1)
            feat_list = exp.as_list(label=0)
            df_exp = pd.DataFrame(feat_list, columns=['feature', 'weight'])
            # plot
            plt.figure(figsize=(6, max(3, 0.35 * len(df_exp))))
            sns.barplot(x='weight', y='feature', data=df_exp, palette='vlag')
            plt.title('LIME explanation (evidence for fake)')
            plt.xlabel('Weight (positive => evidence for fake)')
            plt.tight_layout()
            if save:
                timestamp = int(time.time())
                explanation_path = os.path.join(plots_dir, f'lime_predict_explain_{timestamp}.png')
                plt.savefig(explanation_path, bbox_inches='tight')
                print('Saved LIME explanation to', explanation_path)
            plt.show()
            out['explanation_df'] = df_exp
            out['explanation_path'] = explanation_path
        except Exception as e:
            print('LIME explanation failed:', e)
    # Optionally append to CSV of predictions
    if save:
        rec = {
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
            'text': text,
            'rf_pred': out.get('rf_pred'),
            'rf_prob_fake': out.get('rf_prob_fake'),
            'rf_prob_real': out.get('rf_prob_real'),
            'bert_pred': out.get('bert_pred'),
            'bert_score': out.get('bert_score'),
            'explanation_path': out.get('explanation_path')
        }
        preds_csv = os.path.join(data_root, 'predictions_log.csv')
        df_rec = pd.DataFrame([rec])
        if os.path.exists(preds_csv):
            df_rec.to_csv(preds_csv, mode='a', header=False, index=False)
        else:
            df_rec.to_csv(preds_csv, index=False)
        print('Appended prediction to', preds_csv)

    return out

# ========== 範例使用方式 ==========
# 1) 直接在此處把要測試的新聞貼入 news_text 並執行 下方呼叫。
news_text = """A suspect who shot at a White House checkpoint was killed in an exchange of fire with Secret Service agents on Saturday evening, officials have confirmed.

The BBC's US media partner CBS has named the suspect as Nasire Best, a 21-year-old man who was known to the protection agency and had a documented history of mental health conditions.

US President Donald Trump thanked the officers for their \"swift and professional action\" in apprehending the gunman, who he said in a Truth Social post had a \"violent history and possible obsession with our Country's most cherished structure\".

The shooting comes only a month after a gunman opened fire at the White House Correspondents' Dinner.

The incident on Saturday remains under investigation.

Shortly before 18:00 local time (23:00 GMT), a man pulled a gun from his bag and \"began firing\" outside the White House at the intersection of 17th Street and Pennsylvania Avenue NW in Washington DC, near the Eisenhower Executive Office Building.

Secret Service officers posted on the corner returned fire, striking the gunman. He was then taken to hospital, where he was pronounced dead.

A bystander was also wounded in the shooting, but the Secret Service did not give further details on their condition. No officers were injured in the attack.

Trump was at the White House at the time, but \"no protectees or operations were impacted\", the agency said.

The suspect was later identified in US media as Best, who had been known to both the Secret Service and the Metropolitan Police Department and had used a revolver, law enforcement sources told CBS.

A source familiar with the investigation told CBS that Best had attempted to gain entry to the White House in July 2025 and had been arrested by officers nearby, after which he spent time at a psychiatric facility. He had been living in Washington DC for 18 months.

\"Thank you to our great Secret Service and Law Enforcement for the swift and professional action taken this evening against a gunman near the White House,\" Trump wrote on social media.

Noting that the shooting had occurred since the White House Correspondents' Dinner was disrupted by a different shooter, he said it showed how important it was \"for all future Presidents, to get, what will be, the most safe and secure space of its kind ever built in Washington\" - a reference to his planned White House ballroom.

After shots were heard, reporters at the White House were rushed into a briefing room. Some had been filming when the incident occurred and shots could be heard in the distance as they spoke to camera.

Footage shared by ABC's senior White House correspondent Selina Wang showed her taking cover as a volley of shots could be heard from across the North Lawn.

\"We were told to sprint to the press briefing room where we are holding now,\" Wang wrote on X.

Aaron Navarro, a CBS News reporter, told the BBC he had been on the North Lawn when he heard gunshots, \"at points sounding like they were coming from different guns, just outside the grounds\".

\"As soon as we heard it, we ducked down and I started to see other reporters starting to run, and you shortly heard Secret Service officers saying 'get inside, get inside',\" he said.

Once inside, he said reporters were locked down in the press briefing room for around 30 minutes. Outside, they saw Secret Service officers and then, just beyond the grounds, they eventually saw ambulances.

Navarro said it was unclear exactly where Trump was inside the White House when the shooting took place and \"whether he even heard it, as it was a good distance [away]\".

He said the shooting took place in a busy area with a cafe and restaurants, but that it was not as busy as it could have been since the shooting occurred on a weekend evening.

Senate Majority Leader John Thune and House Speaker Mike Johnson praised the Secret Service for their \"decisive action\" in responding to the shooting.

Thune wrote on a social media that he was \"grateful\" for their efforts, while Johnson said on X: \"Our prayers are with the victims of tonight's senseless shooting for a speedy recovery.\""""

# 2) 呼叫函式（替換 news_text 為你要測試的字串）
res = predict_and_explain(news_text)